In [ ]:
!nvcc --version
!pip install git+https://github.com/andreinechaev/nvcc4jupyter
%load_ext nvcc4jupyter

In [ ]:
%%cuda
#include <stdio.h>

__global__ void op1(int a[], int alen, int t_trds) {
    int tid = threadIdx.x;
    for (int i=tid;i<alen;i=i+t_trds) {
        a[i]=i*i;
    }
}

__global__ void op2(int a[], int alen, int t_trds) {
    int tid = threadIdx.x;
    for (int i=tid;i<alen;i=i+t_trds) {
        a[i]=i*i*i;
    }
}

__global__ void op3(int a[], int b[], int alen, int t_trds) {
    int tid = threadIdx.x;
    for (int i=tid;i<alen;i=i+t_trds) {
        b[i]=a[i]+b[i];
    }
}

int main(){
    int a[3200], *da;
    int b[3200], *db;
    for(int i=0;i<3200;i++) {
        a[i]=i+1;
        b[i]=i+1;
    }
    cudaMalloc(&da, 3200*sizeof(int));
    cudaMalloc(&db, 3200*sizeof(int));
    cudaMemcpy(da,a,3200*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(db,b,3200*sizeof(int), cudaMemcpyHostToDevice);
    op1<<<1,32>>>(da, 3200, 32);
    op2<<<1,32>>>(db, 3200, 32);
    op3<<<1,32>>>(da, db, 3200, 32);
    cudaMemcpy(b,db,3200*sizeof(int), cudaMemcpyDeviceToHost);

}

In [ ]:
%%cuda
#include <stdio.h>
#include <math.h>

__global__ void eucd(int a[], int N, double m[]) {
    int tid = blockIdx.x*1024+threadIdx.x;
    int col = tid%N;
    int row = tid/N;
    if(row<N && col<N) {
        double dx = a[row*2] - a[col*2];
        double dy = a[row*2+1] - a[col*2+1];
        m[row*N+col] = sqrt((dx * dx) + (dy * dy));
    }
}

int main(){
    int N = 5000;
    int *vector, *hvector;
    double *mvector, *cvector;
    cudaMalloc(&vector, N * 2 * sizeof(int));
    cudaMalloc(&mvector, N * N * sizeof(double));
    hvector = (int*)malloc(N*2*sizeof(int));
    cvector = (double*)malloc(N*N*sizeof(double));
    int pairs = N*N;
    for(int i=0;i<N;i++) {
        hvector[i*2] = i*2;
        hvector[i*2+1] = i*2+1;
    }
    cudaMemcpy(vector,hvector,N*2*sizeof(int), cudaMemcpyHostToDevice);
    int nblocks = ceil((pairs*1.0)/(1024*1.0));
    eucd<<<nblocks,1024>>>(vector, N, mvector);
    cudaMemcpy(cvector,mvector,N*N*sizeof(double), cudaMemcpyDeviceToHost);
    double maxi = 0;
    for(int i=0;i<N*N;i++) {
        if(cvector[i]>maxi) {
            maxi=cvector[i];
        }
    }
    printf("%f", maxi);
}